# Day 3：Tool Calling 鲁棒性

今天的目标是理解 Agent 中工具调用的核心问题：**如何让 LLM 稳定、可靠地调用工具？**

Agent 面试中最常见的问题之一就是："你的 Agent 调工具失败了怎么办？"

今天的学习路线：
1. Pydantic 定义严格的 JSON Schema → 约束输入/输出格式
2. 结构化输出方案对比 → Function Calling / JSON Mode / Instructor
3. 优雅降级策略 → 超时、格式错误、参数缺失如何处理
4. 中间件设计模式 → 在工具调用前后加拦截器
5. 动手实现三个基础工具 → file_read、shell_exec、python_run

## 一、Pydantic 定义严格的 JSON Schema

工具的输入/输出必须有类型约束，不能让 LLM 自由发挥。

Pydantic 的 `Field(description=...)` 会自动转为 JSON Schema 注入 Prompt，告诉 LLM 每个字段的含义。

这是一切鲁棒性的基础——只有 Schema 够严格，后面才能做校验和纠错。

In [ ]:
from pydantic import BaseModel, Field, model_validator
from typing import Optional, Literal

class FileReadInput(BaseModel):
    path: str = Field(description="文件的绝对路径")
    encoding: str = Field(default="utf-8", description="文件编码")
    mode: Literal["read", "tail"] = Field(default="read", description="读取模式")
    lines: Optional[int] = Field(default=None, description="tail 模式下的行数")
    
    @model_validator(mode='after')
    def check_tail_lines(self):
        """交叉字段校验：tail 模式必须指定 lines"""
        if self.mode == "tail" and self.lines is None:
            raise ValueError("tail 模式必须指定 lines")
        return self

# 查看自动生成的 JSON Schema——这就是注入到 LLM Prompt 中的内容
print(FileReadInput.model_json_schema())

**要点总结：**
- `Field(description=...)` 是关键的提示工程——写得好不好直接影响 LLM 能否正确调用
- `model_validator` 可以做交叉字段校验，比如"tail 模式必须指定行数"
- JSON Schema 会自动注入到 LLM 的 tool/function 描述中，告诉它每个字段是什么、怎么填

### 实战：定义完整的工具集 Schema

一个好的工具 Schema 应该包含：
- **必填参数**：没有默认值的字段，LLM 必须提供
- **可选参数**：有默认值的字段，LLM 可以省略
- **枚举约束**：用 Literal 限制可选值，防止 LLM 瞎编
- **描述文字**：清晰的 description，帮助 LLM 理解参数含义

下面定义三个基础工具的 Schema：

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional, Literal

# ===== 工具1：文件读取 =====
class FileReadInput(BaseModel):
    """读取文件内容的工具"""
    path: str = Field(description="文件的绝对路径，例如 /home/user/README.md")
    encoding: str = Field(default="utf-8", description="文件编码，默认 utf-8")
    max_lines: Optional[int] = Field(default=None, description="最多读取的行数，不指定则读取全部")

# ===== 工具2：Shell 执行 =====
class ShellExecInput(BaseModel):
    """执行 Shell 命令的工具"""
    command: str = Field(description="要执行的 Shell 命令，注意安全性")
    timeout: int = Field(default=30, description="超时时间（秒），默认 30 秒")
    work_dir: Optional[str] = Field(default=None, description="工作目录，默认当前目录")

# ===== 工具3：Python 代码运行 =====
class PythonRunInput(BaseModel):
    """运行 Python 代码的工具"""
    code: str = Field(description="要执行的 Python 代码，必须是完整可执行的")
    timeout: int = Field(default=30, description="超时时间（秒），默认 30 秒")
    language: Literal["python"] = Field(default="python", description="代码语言")

# 打印所有工具的 Schema
for name, cls in [("FileReadInput", FileReadInput), ("ShellExecInput", ShellExecInput), ("PythonRunInput", PythonRunInput)]:
    print(f"\n===== {name} =====")
    print(cls.model_json_schema())

## 二、结构化输出方案对比

让 LLM 输出结构化数据（如工具调用的参数），有四种主流方案：

| 方案 | 原理 | 优点 | 缺点 |
|------|------|------|------|
| **Function Calling** | API 原生支持，模型训练时已对齐 | 最稳定，格式正确率最高 | 需要 API 支持，不是所有模型都有 |
| **JSON Mode** | 强制输出合法 JSON | 简单直接 | 不保证字段完整，可能存在多余字段 |
| **Instructor 库** | 用 Pydantic 模型约束输出，自动重试 | 灵活，可以自定义校验 | 多一层依赖，需要额外配置 |
| **正则提取** | 从任意文本中提取 JSON | 万能，什么模型都能用 | 脆弱，格式稍微变化就失败 |

**推荐策略：**Function Calling > Instructor > JSON Mode > 正则提取

在实际项目中，通常用 Function Calling 做主路径，正则提取做兜底。

### 方案一：Function Calling（推荐，最稳定）

OpenAI / Anthropic 原生支持。LangChain 封装了 `bind_tools` 方法，直接把 Pydantic 模型转为工具定义。

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field

# 用 Pydantic 定义工具 Schema
class GetWeather(BaseModel):
    """查询指定城市的天气"""
    city: str = Field(description="城市名称，例如 '北京' 或 'shanghai'")
    unit: str = Field(default="celsius", description="温度单位：celsius 或 fahrenheit")

# 初始化 LLM 并绑定工具（bind_tools 会自动将 Pydantic schema 转为 JSON Schema）
llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    api_key=os.getenv("MIMO_API_KEY"),
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
    temperature=0,
)

# bind_tools 的原理：将工具 Schema 注入到 LLM 的 system prompt 中
# LLM 收到用户消息后，会自己判断是否需要调用工具
llm_with_tools = llm.bind_tools([GetWeather])

# 测试：模型应该识别出「查询天气」需要调用工具
response = llm_with_tools.invoke([HumanMessage(content="北京今天天气怎么样？")])
print("模型返回的消息类型:", type(response).__name__)
print("是否有工具调用:", bool(response.tool_calls))
if response.tool_calls:
    print("工具调用详情:")
    for tc in response.tool_calls:
        print(f"  工具名: {tc['name']}")
        print(f"  参数: {tc['args']}")

### 方案二：JSON Mode

强制 LLM 输出合法 JSON。LangChain 提供了 `with_structured_output` 方法：

In [ ]:
from langchain_core.messages import HumanMessage

# with_structured_output 会让 LLM 返回一个 Pydantic 模型实例（而非普通 AI 消息）
# 底层原理：在 prompt 末尾加上 "You must output a JSON object that matches this schema..."
structured_llm = llm.with_structured_output(GetWeather)

# 这回直接返回 Pydantic 对象，不是 AI 消息
result = structured_llm.invoke("查询上海明天的温度，使用华氏度")
print("类型:", type(result))
print("结果:", result)
print(f"城市: {result.city}, 单位: {result.unit}")

### 方案三：Instructor 库

Instructor 用 Pydantic 模型约束 LLM 输出，失败时自动重试。适合对格式要求极高且模型不够稳定的场景。

**注意：** 需要 `pip install instructor`。如果你的 LLM 已经支持 Function Calling，用方案一就够了。

In [ ]:
# Instructor 的使用示例（需要先 pip install instructor）
# import instructor
# client = instructor.from_openai(
#     ChatOpenAI(
#         model="mimo-v2.5-pro",
#         api_key=os.getenv("MIMO_API_KEY"),
#         base_url="https://token-plan-cn.xiaomimimo.com/v1",
#     )
# )
# 
# result = client.chat.completions.create(
#     response_model=GetWeather,
#     messages=[{"role": "user", "content": "查看北京天气"}],
#     max_retries=2,  # Instructor 的核心价值：自动重试
# )
# print(result)

### 方案四：正则提取（最后手段）

当模型不支持 Function Calling 也不支持 JSON Mode 时，用正则从文本中挖出 JSON。
脆弱但万能——给任何模型都能用。

In [ ]:
import re
import json

def extract_json_from_text(text: str):
    """从任意文本中提取 JSON 对象——最后手段"""
    # 尝试匹配 ```json ... ``` 代码块
    match = re.search(r'```json\s*(\{.*?\})\s*```', text, re.DOTALL)
    if match:
        return json.loads(match.group(1))
    
    # 尝试匹配裸 JSON 对象
    match = re.search(r'\{.*?\}', text, re.DOTALL)
    if match:
        return json.loads(match.group(0))
    
    raise ValueError(f"无法从文本中提取 JSON: {text[:100]}...")

# 测试
test_text = '好的，以下是天气信息：\n```json\n{"city": "北京", "unit": "celsius"}\n```\n希望对你有帮助！'
result = extract_json_from_text(test_text)
print("提取结果:", result)

## 三、优雅降级策略

工具调用可能失败，一个好的 Agent 不会直接崩溃，而是有**多种降级策略**：

| 失败场景 | 降级策略 |
|---------|---------|
| **超时** | 设置 30s 超时，超时后返回错误信息让 LLM 决定下一步 |
| **格式错误** | 自动检测 JSON parse 失败，触发重新生成（最多 2 次） |
| **参数缺失** | 检查必填字段，缺失时让 LLM 补充而不是直接报错 |
| **工具不存在** | 返回可用工具列表，引导 LLM 选择正确的工具 |

下面实现一个完整的工具调用中间件，覆盖上述所有场景。

### 3.1 超时处理

工具执行可能因为网络问题、资源问题卡住。必须设置超时。

In [ ]:
import time
import subprocess
from typing import Any

class ToolTimeoutError(Exception):
    """工具执行超时异常"""
    pass

def execute_with_timeout(func, timeout: float = 30.0, *args, **kwargs):
    """
    带超时的工具执行包装器
    
    方案说明：Python 的 subprocess 可以设置 timeout，但对于纯 Python 函数，
    需要用到 signal（Linux）或 threading（跨平台）。这里演示跨平台的 threading 方案。
    """
    import threading
    
    result_container = {"result": None, "error": None, "done": False}
    
    def wrapper():
        try:
            result_container["result"] = func(*args, **kwargs)
        except Exception as e:
            result_container["error"] = str(e)
        finally:
            result_container["done"] = True
    
    thread = threading.Thread(target=wrapper, daemon=True)
    thread.start()
    thread.join(timeout=timeout)
    
    if not result_container["done"]:
        return {
            "success": False,
            "error": f"工具执行超时（>{timeout}s）",
            "error_type": "timeout"
        }
    
    if result_container["error"]:
        return {
            "success": False,
            "error": result_container["error"],
            "error_type": "execution_error"
        }
    
    return {
        "success": True,
        "result": result_container["result"]
    }

# 测试超时
def long_running_task():
    time.sleep(5)
    return "完成"

print("正常执行:", execute_with_timeout(long_running_task, timeout=10))
print("超时执行:", execute_with_timeout(long_running_task, timeout=1))

### 3.2 格式错误自动重试

LLM 输出的 JSON 可能不合法（缺少引号、多了逗号、嵌套错误等）。需要自动检测并触发重新生成。

In [ ]:
import json
from langchain_core.messages import HumanMessage, AIMessage


class ToolCallValidator:
    """工具调用校验器：自动检测格式错误并触发重试"""
    
    def __init__(self, max_retries: int = 2):
        self.max_retries = max_retries
    
    def validate_and_retry(self, llm_response: AIMessage, tool_schema: type[BaseModel], llm) -> dict:
        """
        校验 LLM 的工具调用参数，失败时自动重试
        
        返回值：
        - {"success": True, "params": <校验后的参数>}
        - {"success": False, "error": <错误信息>}
        """
        # 情况1：LLM 没有返回工具调用
        if not llm_response.tool_calls:
            return {"success": False, "error": "LLM 没有返回工具调用", "error_type": "no_tool_call"}
        
        errors = []
        
        for attempt in range(self.max_retries + 1):
            try:
                # 尝试用 Pydantic 校验参数
                tool_call = llm_response.tool_calls[0]
                params = tool_schema(**tool_call["args"])
                return {"success": True, "params": params}
                
            except Exception as e:
                errors.append(str(e))
                
                if attempt < self.max_retries:
                    # 构建错误反馈，让 LLM 知道哪里错了
                    error_feedback = f"""
你的上一次工具调用参数格式有误：{e}

请根据以下 Schema 修正参数：
{tool_schema.model_json_schema()}

只输出修正后的工具调用，不要输出其他内容。
"""
                    llm_response = llm.invoke([
                        HumanMessage(content=error_feedback)
                    ])
        
        return {
            "success": False,
            "error": f"重试 {self.max_retries} 次后仍然失败，错误: {errors}",
            "error_type": "validation_failed",
            "attempt_errors": errors
        }

# 测试校验器
validator = ToolCallValidator(max_retries=2)
print("校验器初始化完成，max_retries =", validator.max_retries)

### 3.3 参数缺失处理

即使 JSON 合法，也可能缺少必填字段。需要检查并让 LLM 补充。

In [ ]:
def check_required_fields(params: dict, schema: type[BaseModel]) -> list[str]:
    """
    检查参数是否满足 Schema 的必填要求
    
    返回缺失的必填字段列表
    """
    schema_json = schema.model_json_schema()
    required_fields = schema_json.get("required", [])
    
    missing = []
    for field in required_fields:
        if field not in params or params[field] is None:
            missing.append(field)
    
    return missing

# 测试
incomplete_params = {"mode": "read"}  # 缺少必填字段 path
missing = check_required_fields(incomplete_params, FileReadInput)
print(f"缺失的必填字段: {missing}")  # 应该输出 ['path']

complete_params = {"path": "/tmp/test.txt", "mode": "read"}
missing = check_required_fields(complete_params, FileReadInput)
print(f"缺失的必填字段: {missing}")  # 应该输出 []

## 四、中间件设计模式

在工具调用前后各加一层拦截器，形成"洋葱模型"：

```
请求 → 前置拦截器(参数校验/权限检查) → 工具执行 → 后置拦截器(结果格式化/错误分类/日志) → 响应
```

这种设计的好处：
- **统一错误处理**：不用在每个工具里写 try/except
- **可观测性**：所有工具调用都有日志，出问题时能回溯
- **可扩展**：新增工具自动继承中间件能力

In [ ]:
import functools
from datetime import datetime
from typing import Callable


class ToolMiddleware:
    """
    工具调用中间件：在工具执行前后各加一层处理
    
    前置拦截：参数校验、权限检查
    后置拦截：结果格式化、错误分类、日志记录
    """
    
    def __init__(self, max_retries: int = 2, timeout: float = 30.0):
        self.max_retries = max_retries
        self.timeout = timeout
        self.call_history = []  # 记录所有工具调用历史
    
    def wrap(self, tool_func: Callable, schema: type[BaseModel], tool_name: str):
        """
        包装一个工具函数，加上中间件能力
        
        参数：
        - tool_func: 原始工具函数
        - schema: 工具的 Pydantic Schema
        - tool_name: 工具名称（用于日志）
        
        返回：包装后的异步安全工具函数
        """
        @functools.wraps(tool_func)
        def wrapper(**kwargs):
            start_time = datetime.now()
            call_record = {
                "tool": tool_name,
                "params": kwargs,
                "start_time": start_time.isoformat(),
            }
            
            try:
                # ===== 前置拦截：参数校验 =====
                validated = schema(**kwargs)
                call_record["validated_params"] = validated.model_dump()
                
                # 检查必填字段
                missing = check_required_fields(kwargs, schema)
                if missing:
                    call_record["success"] = False
                    call_record["error"] = f"缺少必填字段: {missing}"
                    call_record["error_type"] = "missing_required_fields"
                    self.call_history.append(call_record)
                    return {
                        "success": False,
                        "error": f"缺少必填字段: {missing}",
                        "error_type": "missing_required_fields"
                    }
                
                # ===== 执行工具（带超时） =====
                result = execute_with_timeout(
                    tool_func, self.timeout, **validated.model_dump()
                )
                
                # ===== 后置拦截：格式化结果 =====
                elapsed = (datetime.now() - start_time).total_seconds()
                call_record["elapsed"] = elapsed
                call_record["success"] = result.get("success", True)
                
                if not result.get("success"):
                    call_record["error"] = result.get("error")
                    call_record["error_type"] = result.get("error_type")
                else:
                    call_record["result"] = str(result.get("result", ""))[:200]  # 截断长结果
                
                self.call_history.append(call_record)
                
                return {
                    "success": True,
                    "data": result,
                    "elapsed": elapsed,
                    "tool": tool_name
                }
                
            except Exception as e:
                call_record["success"] = False
                call_record["error"] = str(e)
                call_record["error_type"] = type(e).__name__
                self.call_history.append(call_record)
                
                return {
                    "success": False,
                    "error": str(e),
                    "error_type": type(e).__name__,
                    "tool": tool_name
                }
        
        # 把 schema 绑到 wrapper 上，方便外部获取
        wrapper.schema = schema
        wrapper.tool_name = tool_name
        return wrapper
    
    def get_history(self):
        """获取工具调用历史——用于调试和可观测性"""
        return self.call_history


# 初始化中间件
middleware = ToolMiddleware(max_retries=2, timeout=10.0)
print("中间件初始化完成")

## 五、实现三个基础工具

把前面学到的所有知识点整合起来，实现三个带完整中间件的工具。

In [ ]:
import subprocess
import sys
from pathlib import Path


# ===== 工具1：文件读取 =====
def tool_file_read(path: str, encoding: str = "utf-8", max_lines: int = None) -> str:
    """读取文件内容"""
    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(f"文件不存在: {path}")
    
    with open(file_path, "r", encoding=encoding) as f:
        if max_lines:
            lines = [next(f) for _ in range(max_lines)]
            return "".join(lines)
        return f.read()


# ===== 工具2：Shell 执行 =====
def tool_shell_exec(command: str, timeout: int = 30, work_dir: str = None) -> dict:
    """执行 Shell 命令（带安全限制）"""
    # 禁止的危险命令
    dangerous_commands = ["rm -rf /", "mkfs", "dd if=", ":(){ :|:& };:"]
    for dc in dangerous_commands:
        if dc in command:
            raise ValueError(f"检测到危险命令: {dc}")
    
    result = subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True,
        timeout=timeout,
        cwd=work_dir or ".",
    )
    
    return {
        "stdout": result.stdout,
        "stderr": result.stderr,
        "return_code": result.returncode
    }


# ===== 工具3：Python 代码运行 =====
def tool_python_run(code: str, timeout: int = 30, language: str = "python") -> dict:
    """在隔离环境中运行 Python 代码"""
    result = subprocess.run(
        [sys.executable, "-c", code],
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    
    return {
        "stdout": result.stdout[:5000],  # 截断输出
        "stderr": result.stderr[:5000],
        "return_code": result.returncode
    }


# ===== 用中间件包装所有工具 =====
wrapped_file_read = middleware.wrap(tool_file_read, FileReadInput, "file_read")
wrapped_shell_exec = middleware.wrap(tool_shell_exec, ShellExecInput, "shell_exec")
wrapped_python_run = middleware.wrap(tool_python_run, PythonRunInput, "python_run")

print("三个工具已通过中间件包装完成")
print(f"  - {wrapped_file_read.tool_name}: {wrapped_file_read.schema.__name__}")
print(f"  - {wrapped_shell_exec.tool_name}: {wrapped_shell_exec.schema.__name__}")
print(f"  - {wrapped_python_run.tool_name}: {wrapped_python_run.schema.__name__}")

### 测试：工具调用与中间件验证

In [ ]:
# 测试1：正常调用 file_read
print("===== 测试1：读取文件 =====")
result = wrapped_file_read(path="../CLAUDE.md", encoding="utf-8")
print(f"成功: {result['success']}")
print(f"耗时: {result.get('elapsed', 'N/A')}s")
if result["success"]:
    print(f"内容预览: {str(result['data']['result'])[:100]}...")
else:
    print(f"错误: {result['error']}")

print("\n===== 测试2：Shell 命令 =====")
result = wrapped_shell_exec(command="echo 'hello from shell'", timeout=5)
print(f"成功: {result['success']}")
if result["success"]:
    print(f"stdout: {result['data']['result']['stdout'].strip()}")

print("\n===== 测试3：Python 代码 =====")
result = wrapped_python_run(code="print('hello from python'); print(sum([1,2,3,4,5]))")
print(f"成功: {result['success']}")
if result["success"]:
    print(f"stdout: {result['data']['result']['stdout'].strip()}")

print("\n===== 测试4：缺少必填参数（应该失败） =====")
result = wrapped_file_read()  # 缺少 path
print(f"成功: {result['success']}")
print(f"错误: {result['error']}")

print("\n===== 调用历史 =====")
for record in middleware.get_history():
    print(f"  {record['tool']} | 成功: {record['success']} | 开始: {record['start_time']}")

## 六、整合：将工具接入 LangGraph Agent

在前两天学习的 LangGraph 基础上，把包装好的工具集成为一个完整的 Tool Calling Agent。

In [ ]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, AnyMessage
from langgraph.graph.message import add_messages
import json


# ===== 1. 定义 Agent 状态 =====
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    retry_count: int   # 当前工具调用重试次数


# ===== 2. 定义可用工具（映射表） =====
TOOLS = {
    "file_read": wrapped_file_read,
    "shell_exec": wrapped_shell_exec,
    "python_run": wrapped_python_run,
}

# ===== 3. LLM 节点 =====
def call_llm(state: AgentState):
    """LLM 思考节点：判断用户意图，决定是否调用工具"""
    llm_with_tools = llm.bind_tools([
        FileReadInput,
        ShellExecInput,
        PythonRunInput,
    ])
    
    response = llm_with_tools.invoke(state["messages"])
    return {
        "messages": [response],
        "retry_count": state.get("retry_count", 0)
    }


# ===== 4. 工具执行节点 =====
def execute_tools(state: AgentState):
    """
    执行 LLM 请求的工具调用
    
    返回值：ToolMessage 列表，包含工具执行结果
    """
    last_msg = state["messages"][-1]
    tool_messages = []
    
    for tool_call in last_msg.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_func = TOOLS.get(tool_name)
        
        if tool_func is None:
            result = {"success": False, "error": f"未知工具: {tool_name}"}
        else:
            result = tool_func(**tool_args)
        
        # 工具返回结果也作为 ToolMessage 回传给 LLM
        tool_messages.append(ToolMessage(
            content=json.dumps(result, ensure_ascii=False),
            tool_call_id=tool_call["id"]
        ))
    
    return {"messages": tool_messages}


# ===== 5. 条件路由：是否需要调用工具？ =====
def should_call_tool(state: AgentState) -> Literal["tools", "end"]:
    """检查 LLM 的最后一条消息是否包含工具调用"""
    last_msg = state["messages"][-1]
    
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        # 检查重试次数是否超限
        if state.get("retry_count", 0) >= 3:
            return "end"  # 超过 3 次重试，停止
        return "tools"
    
    return "end"


# ===== 6. 构建状态图 =====
builder = StateGraph(AgentState)
builder.add_node("llm", call_llm)
builder.add_node("tools", execute_tools)

builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", should_call_tool, {"tools": "tools", "end": END})
builder.add_edge("tools", "llm")  # 工具执行完回到 LLM，形成 ReAct 循环

# 编译
agent = builder.compile()

print("Agent 构建完成！")
print(f"可用工具: {list(TOOLS.keys())}")

### 运行 Agent

测试一下 Agent 能否正确处理带工具调用的任务。

In [ ]:
# 测试：让 Agent 读取文件
result = agent.invoke({
    "messages": [HumanMessage(content="读取 ../README.md 文件的前 20 行")],
    "retry_count": 0
})

# 打印对话历史
for msg in result["messages"]:
    role = type(msg).__name__.replace("Message", "")
    content = msg.content if hasattr(msg, "content") else str(msg)
    print(f"[{role}] {str(content)[:150]}...")
    print("---")

## 今日总结

今天完成了 Tool Calling 鲁棒性的完整学习：

1. **Pydantic Schema** — 约束工具输入/输出格式，是所有鲁棒性的基础
2. **结构化输出四种方案** — Function Calling（首选）> Instructor > JSON Mode > 正则提取
3. **优雅降级策略** — 超时处理、格式错误重试、参数缺失检查、工具不存在引导
4. **中间件设计** — 前置拦截 + 后置拦截，统一错误处理和日志记录
5. **三个基础工具** — file_read、shell_exec、python_run 带完整中间件
6. **LangGraph 集成** — 将工具接入状态图，形成 LLM ↔ Tools 的 ReAct 循环

**面试要点：** 面试官问"你的 Agent 调工具失败了怎么办"时，你应该能回答：
- 格式错误：Pydantic 自动校验 + 最多 2 次重试
- 超时：所有工具有 30s 超时保护
- 参数缺失：check_required_fields 检查 + 引导 LLM 补充
- 工具不存在：返回可用工具列表
- 以上策略通过中间件统一管理，所有工具自动继承

明天学习 MCP 协议——Anthropic 推出的 Agent 工具协议标准，让你的工具可以被任何 LLM 框架调用。